# SSDC 2026 — Exploratory Data Analysis (EDA): Student Placement System Dataset by kata edward

## Bagian 0 - Setup & Sanity Check

In [135]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### 0.1 - Konfirmasi Shape

In [136]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = "../data/clean/"

company = pd.read_csv(
    DATA_DIR + "company_clean.csv",
    dtype={"id_company": str, "pic_phone": str},
    parse_dates=["created_at"],
)

status_student = pd.read_csv(
    DATA_DIR + "status_student_clean.csv",
    dtype={"id_status": str, "NIM": str, "no_whatsapp": str},
    parse_dates=["sync_date"],
)

student_all = pd.read_csv(
    DATA_DIR + "student_all_clean.csv", dtype={"NIM": str, "hp": str}
)

talent_request = pd.read_csv(
    DATA_DIR + "talent_request_clean.csv",
    dtype={"id_talent_req": str, "id_company": str, "no_whatsapp": str},
    parse_dates=["request_date"],
)

tracking_company = pd.read_csv(
    DATA_DIR + "tracking_company_clean.csv",
    dtype={
        "id_tracking_company": str,
        "id_talent_req": str,
        "id_company": str,
        "list_nim": str,
    },
    parse_dates=["request_date", "send_date"],
)

tracking_student = pd.read_csv(
    DATA_DIR + "tracking_student_clean.csv",
    dtype={"id_tracking_student": str, "NIM": str, "id_tracking_company": str},
    parse_dates=["last_update"],
)

tables = {
    "company": company,
    "talent_request": talent_request,
    "student_all": student_all,
    "status_student": status_student,
    "tracking_company": tracking_company,
    "tracking_student": tracking_student,
}

expected_shapes = {
    "company": 1500,
    "status_student": 25000,
    "student_all": 25000,
    "talent_request": 12000,
    "tracking_company": 12000,
    "tracking_student": 41600,
}

for name in tables:
    n = tables[name].shape[0]
    status = "MISMATCH"
    if n == expected_shapes[name]:
        status = "OK"
    print(
        f"{name:18s} shape={n}  expected_rows={expected_shapes.get(name):<6} [{status}]"
    )

company            shape=1500  expected_rows=1500   [OK]
talent_request     shape=12000  expected_rows=12000  [OK]
student_all        shape=25000  expected_rows=25000  [OK]
status_student     shape=25000  expected_rows=25000  [OK]
tracking_company   shape=12000  expected_rows=12000  [OK]
tracking_student   shape=41600  expected_rows=41600  [OK]


### 0.2 - Sanity Check Ringkas (PK, FK, tanggal)

In [137]:
pk_checks = {
    "company": "id_company",
    "status_student": "id_status",
    "student_all": "NIM",
    "talent_request": "id_talent_req",
    "tracking_company": "id_tracking_company",
    "tracking_student": "id_tracking_student",
}

print("-- Primary key unik -- ")
for name, pk in pk_checks.items():
    df = tables[name]
    dup = df[pk].duplicated().sum()
    print(f"{name:18s} PK={pk:20s} duplikat={dup} [{'OK' if dup==0 else 'ANOMALI'}]")


print("\n-- Foreign key yatim (orphan) --")

# fmt: off
orphan_ss_nim = ~status_student["NIM"].isin(student_all["NIM"])
orphan_tr_company = ~talent_request["id_company"].isin(company["id_company"])
orphan_tc_talent = ~tracking_company["id_talent_req"].isin(talent_request["id_talent_req"])
orphan_tc_company = ~tracking_company["id_company"].isin(company["id_company"])
orphan_ts_tc = ~tracking_student["id_tracking_company"].isin(tracking_company["id_tracking_company"])
orphan_ts_nim = ~tracking_student["NIM"].isin(student_all["NIM"])

print(f"status_student.NIM -> student_all.NIM : {orphan_ss_nim.sum()} yatim")
print(f"talent_request.id_company -> company.id_company : {orphan_tr_company.sum()} yatim")
print(f"tracking_company.id_talent_req -> talent_request.id_talent_req : {orphan_tc_talent.sum()} yatim")
print(f"tracking_company.id_company -> company.id_company : {orphan_tc_company.sum()} yatim")
print(f"tracking_student.id_tracking_company -> tracking_company.id_tracking_company : {orphan_ts_tc.sum()} yatim")
print(f"tracking_student.NIM -> student_all.NIM : {orphan_ts_nim.sum()} yatim")
# fmt: on


print("\n-- Draft tracking_company (expected 598) --")
draft_mask = tracking_company["send_date"].isna() & tracking_company["list_nim"].isna()
print(f"Baris tanpa send_date & list_nim: {draft_mask.sum()}")
print(tracking_company.loc[draft_mask, "progress"].value_counts())


print("\n-- Rentang tanggal --")
date_cols = {
    "company.created_at": company["created_at"],
    "talent_request.request_date": talent_request["request_date"],
    "status_student.sync_date": status_student["sync_date"],
    "tracking_company.request_date": tracking_company["request_date"],
    "tracking_company.send_date": tracking_company["send_date"],
    "tracking_student.last_update": tracking_student["last_update"],
}

for label, col in date_cols.items():
    print(f"{label:30s} min={col.min().date()}  max={col.max().date()}")

sent = tracking_company.dropna(subset=["send_date"])
invalid_order = (sent["send_date"] < sent["request_date"]).sum()
print(f"\ntracking_company.send_date < tracking_company.request_date (harus 0): {invalid_order}")  # fmt: skip

-- Primary key unik -- 
company            PK=id_company           duplikat=0 [OK]
status_student     PK=id_status            duplikat=0 [OK]
student_all        PK=NIM                  duplikat=0 [OK]
talent_request     PK=id_talent_req        duplikat=0 [OK]
tracking_company   PK=id_tracking_company  duplikat=0 [OK]
tracking_student   PK=id_tracking_student  duplikat=0 [OK]

-- Foreign key yatim (orphan) --
status_student.NIM -> student_all.NIM : 0 yatim
talent_request.id_company -> company.id_company : 0 yatim
tracking_company.id_talent_req -> talent_request.id_talent_req : 0 yatim
tracking_company.id_company -> company.id_company : 0 yatim
tracking_student.id_tracking_company -> tracking_company.id_tracking_company : 0 yatim
tracking_student.NIM -> student_all.NIM : 0 yatim

-- Draft tracking_company (expected 598) --
Baris tanpa send_date & list_nim: 598
progress
Draft    598
Name: count, dtype: int64

-- Rentang tanggal --
company.created_at             min=2022-01-02  max=2024-12

### 0.2b - Cross-check STUDENT ALL ↔ STATUS STUDENT (BT-08)

In [138]:
merged = student_all.merge(
    status_student[["NIM", "nama", "program_studi", "semester"]],
    on="NIM",
    how="outer",
    suffixes=("_studentall", "_status"),
    indicator=True,
)

# fmt: off
orphan_left = (merged['_merge'] == 'left_only').sum()   # ada di student_all, tidak di status_student
orphan_right = (merged['_merge'] == 'right_only').sum() # ada di status_student, tidak di student_all
print(f"NIM hanya di student_all (orphan)   : {orphan_left}")
print(f"NIM hanya di status_student (orphan): {orphan_right}")

both = merged[merged['_merge'] == 'both'].copy()
mismatch_nama = (both['nama_studentall'] != both['nama_status']).sum()
mismatch_prodi = (both['program_studi_studentall'] != both['program_studi_status']).sum()
mismatch_semester = (both['semester_studentall'] != both['semester_status']).sum()

print(f"\nDari {len(both)} NIM yang match di kedua tabel:")
print(f"  mismatch nama         : {mismatch_nama}")
print(f"  mismatch program_studi: {mismatch_prodi}")
print(f"  mismatch semester     : {mismatch_semester}")

kesimpulan = "SINKRON 100%" if (orphan_left + orphan_right + mismatch_nama + mismatch_prodi + mismatch_semester) == 0 else "ADA ANOMALI"
print(f"\n>>> Kesimpulan 0.2b: {kesimpulan}")
# fmt: on

NIM hanya di student_all (orphan)   : 0
NIM hanya di status_student (orphan): 0

Dari 25000 NIM yang match di kedua tabel:
  mismatch nama         : 0
  mismatch program_studi: 0
  mismatch semester     : 0

>>> Kesimpulan 0.2b: SINKRON 100%


## Bagian 1 - Profil Dasar Tiap Tabel -> H1